##  Final Model Evaluation with Tuned Thresholds (Youden's J)

In this step, we apply the **custom thresholds** (identified via Youden's J statistic during threshold tuning) to each of our final logistic regression models. 

Instead of the default `0.5` threshold, we use optimized thresholds to improve **Recall**, which is critical for **early screening**.

###  What This Code Does:
- Loads the saved model pipeline and held-out test set.
- Applies the **best-performing threshold**:
  - High BP → 0.23  
  - Diabetes → 0.24  
  - Cardiovascular Condition → 0.24
- Computes:
  - **Confusion Matrix**
  - **Precision / Recall / F1-score** at the selected threshold
- Saves the classification report as CSV files for each target condition.

>  Note: **ROC AUC does not change**, as it is threshold-independent. This step only changes the **decision boundary** for classification.

This is the final evaluation step before deploying our models for real-world use.


In [1]:
import os
import pandas as pd

# --- Path Config ---
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")
STATS_DIR = os.path.join(OUTPUTS_DIR, "statistics")
PLOTS_DIR = os.path.join(OUTPUTS_DIR, "plots")
METRICS_DIR = os.path.join(OUTPUTS_DIR, "metrics")

# Make sure subfolders exist
os.makedirs(STATS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

part19_metrics_dir = os.path.join(METRICS_DIR, "part19_custom_thresholds")
os.makedirs(part19_metrics_dir, exist_ok=True)

part19_plots_dir = os.path.join(PLOTS_DIR, "part19_custom_thresholds")
os.makedirs(part19_plots_dir, exist_ok=True)



In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import os

# === Evaluation Function with Confusion Matrix Plotting ===
def evaluate_with_threshold(label, model_file, test_file, threshold, target_col):

    print(f"\n Evaluating: {label} at threshold = {threshold:.2f}")
    
    # Load model and data
    model = joblib.load(model_file)
    df = pd.read_csv(test_file)
    
    # Separate features and target
    X = df.drop(columns=[target_col])
    y_true = df[target_col].map({"No": 0, "Yes": 1})

    # Predict probabilities
    y_proba = model.predict_proba(X)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    # === Classification Report & Confusion Matrix ===
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    # Save report CSV
    report_df = pd.DataFrame(report).transpose()
    report_path = os.path.join(part19_metrics_dir, f"classification_report_{label.lower().replace(' ', '_')}.csv")
    report_df.to_csv(report_path)
    print(f" Saved classification report to: {report_path}")

    # === Confusion Matrix Plot ===
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
    plt.title(f"Confusion Matrix - {label} (Threshold = {threshold:.2f})")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    cm_path = os.path.join(part19_plots_dir, f"confusion_matrix_{label.lower().replace(' ', '_')}.png")
    plt.savefig(cm_path)
    plt.close()
    print(f" Saved confusion matrix plot to: {cm_path}")

    # Print summary
    print(" Confusion Matrix:")
    print(cm)
    print("\n Classification Summary:")
    print(report_df.loc[["0", "1", "macro avg", "weighted avg"]])


In [3]:
# === Configuration ===
model_paths = {
    "High BP": {
        "target_col": "Has a high blood pressure",
        "model_file": os.path.join(METRICS_DIR, "part14_final_models", "model_highbp.pkl"),
        "test_file": os.path.join(STATS_DIR, "part14_final_models", "test_highbp.csv"),
        "threshold": 0.23
    },
    "Diabetes": {
        "target_col": "Has diabetes",
        "model_file": os.path.join(METRICS_DIR, "part14_final_models", "model_diabetes.pkl"),
        "test_file": os.path.join(STATS_DIR, "part14_final_models", "test_diabetes.csv"),
        "threshold": 0.24
    },
    "Cardio": {
        "target_col": "Cardiovascular condition (Heart disease or stroke)",
        "model_file": os.path.join(METRICS_DIR, "part14_final_models", "model_cardio.pkl"),
        "test_file": os.path.join(STATS_DIR, "part14_final_models", "test_cardio.csv"),
        "threshold": 0.24
    }
}
# === Run for all models ===
for label, details in model_paths.items():
    evaluate_with_threshold(
        label=label,
        model_file=details["model_file"],
        test_file=details["test_file"],
        threshold=details["threshold"],
        target_col=details["target_col"]
    )



 Evaluating: High BP at threshold = 0.23
 Saved classification report to: d:\Projects\health-risk-prediction\outputs\metrics\part19_custom_thresholds\classification_report_high_bp.csv
 Saved confusion matrix plot to: d:\Projects\health-risk-prediction\outputs\plots\part19_custom_thresholds\confusion_matrix_high_bp.png
 Confusion Matrix:
[[5938 8549]
 [ 250 5614]]

 Classification Summary:
              precision    recall  f1-score  support
0              0.959599  0.409885  0.574414  14487.0
1              0.396385  0.957367  0.560643   5864.0
macro avg      0.677992  0.683626  0.567528  20351.0
weighted avg   0.797313  0.567638  0.570446  20351.0

 Evaluating: Diabetes at threshold = 0.24
 Saved classification report to: d:\Projects\health-risk-prediction\outputs\metrics\part19_custom_thresholds\classification_report_diabetes.csv
 Saved confusion matrix plot to: d:\Projects\health-risk-prediction\outputs\plots\part19_custom_thresholds\confusion_matrix_diabetes.png
 Confusion Matrix: